# 02 - Compute NAO Index

## Objective
Compute a box-based NAO index from CanCM4 decadal sea level pressure (`psl`) data.

## Method
- Select Azores and Iceland pressure boxes
- Compute spatial mean pressure in each box
- Select DJFM months from year 2 to year 9
- Define NAO as Azores pressure minus Iceland pressure

## Output
- `df_can_nao`: DataFrame containing NAO index and NAO anomalies for each initialization year and ensemble member

In [ ]:
import pandas as pd
import xarray as xr
import numpy as np

In [ ]:
# Define NAO region boundaries using 0-360 longitude convention

# Azores box
azores_lat_min, azores_lat_max = 36, 40
azores_lon_min, azores_lon_max = 332, 340

# Iceland box
iceland_lat_min, iceland_lat_max = 63, 70
iceland_lon_min, iceland_lon_max = 335, 344

## Compute NAO index

In [ ]:
# Store NAO results here
nao_data = []

# Loop over all initialization years and ensemble members
for year in range(1960, 2006):
    for r in range(1, 11):
        var_name = f"can_decadal_{year}_r{r}"

        # Check whether the dataset exists in the dictionary
        if var_name in decadal_datasets:
            ds = decadal_datasets[var_name]

            try:
                # Ensure longitude is in 0-360 format for regional selection
                ds = ds.assign_coords(lon=((ds.lon + 360) % 360))

                # Define the DJFM period from year 2 to year 9
                start_date = f"{year+1}-12-01"
                end_date = f"{year+9}-03-31"

                # Select sea level pressure over the Azores region and convert Pa to hPa
                psl_azores = ds["psl"].sel(
                    lat=slice(azores_lat_min, azores_lat_max),
                    lon=slice(azores_lon_min, azores_lon_max)
                ) / 100

                # Select sea level pressure over the Iceland region and convert Pa to hPa
                psl_iceland = ds["psl"].sel(
                    lat=slice(iceland_lat_min, iceland_lat_max),
                    lon=slice(iceland_lon_min, iceland_lon_max)
                ) / 100

                # Compute spatial mean pressure in each region
                azores_mean = psl_azores.mean(dim=["lat", "lon"], skipna=True)
                iceland_mean = psl_iceland.mean(dim=["lat", "lon"], skipna=True)

                # Ensure time is in datetime format. Redundant but need to be sure
                azores_mean = azores_mean.assign_coords(time=pd.to_datetime(azores_mean.time.values))
                iceland_mean = iceland_mean.assign_coords(time=pd.to_datetime(iceland_mean.time.values))

                # Select only the requested period: 2-9 year window for each year/member
                azores_subset = azores_mean.sel(time=slice(start_date, end_date))
                iceland_subset = iceland_mean.sel(time=slice(start_date, end_date))

                # Keep only DJFM months: December, January, February, March
                azores_filtered = azores_subset.where(
                    azores_subset["time"].dt.month.isin([12, 1, 2, 3]), drop=True
                )
                iceland_filtered = iceland_subset.where(
                    iceland_subset["time"].dt.month.isin([12, 1, 2, 3]), drop=True
                )

                # Compute the average DJFM pressure over the selected 2-9 year window for each year/member
                psl_djfm_azores_mean = azores_filtered.mean(skipna=True).values
                psl_djfm_iceland_mean = iceland_filtered.mean(skipna=True).values

                # Define NAO index as Azores pressure minus Iceland pressure
                nao_index = psl_djfm_azores_mean - psl_djfm_iceland_mean

                # Store result: NAO index over the selected 2-9 year window for each year/member
                nao_data.append([year, r, nao_index])

            except Exception as e:
                print(f"Skipping {var_name} due to error: {e}")
        else:
            print(f"{var_name} not found in decadal_datasets")

## Create NAO table

In [ ]:
# Convert results into a DataFrame with columns Year, Realizations, NAO Index
df_can_nao = pd.DataFrame(nao_data, columns=["Year", "Realization", "NAO Index"])

# Create member labels: r1, r2, ..., r10
df_can_nao["member"] = "r" + df_can_nao["Realization"].astype(str)

# Compute mean NAO for each realization across all years
overall_nao_mean = df_can_nao.groupby("member")["NAO Index"].mean()

# Compute NAO anomalies by subtracting the mean of the corresponding realization
df_can_nao["NAO Anomaly"] = df_can_nao["NAO Index"] - df_can_nao["member"].map(overall_nao_mean)

# Use Year and member ID as index
df_can_nao.set_index(["Year", "member"], inplace=True)

## Quick check

In [ ]:
print(df_can_nao.head())
print("\nShape:")
print(df_can_nao.shape)

print("\nMean NAO by realization:")
print(overall_nao_mean)